# Phase 3 — The Core Class-Imbalance Experiment

**Question.** Which class-imbalance strategy generalises best for credit-card
fraud detection, and how much does the choice actually matter?

**Design.** Four strategies (no resampling, SMOTE, random undersampling,
class weighting) x three models (Logistic Regression, Random Forest,
XGBoost) x two split protocols (stratified, chronological). Every cell is
retuned in its own right, so no strategy is handicapped by hyperparameters
chosen for another. 5-fold cross-validated, PR-AUC as the headline metric.

**Leakage control.** Resampling lives inside an `imblearn` Pipeline, which
runs a sampler only during `fit` and skips it during `predict`. Validation
and test rows are therefore never oversampled, never dropped, and never used
to synthesise SMOTE neighbours. See `src/fraud/resampling.py` and the 18
tests in `tests/test_resampling.py`.

This notebook only *reads* results — it refits nothing. Metrics come from
`results/imbalance_experiment.csv` and the PR curves are rebuilt from the
test-set scores saved under `results/scores/` by
`scripts/run_imbalance_experiment.py`.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

sys.path.insert(0, str(Path.cwd().parent / "src"))
from fraud import config, data, experiment  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

MODEL_ORDER = ["logistic_regression", "random_forest", "xgboost"]
STRATEGY_ORDER = ["none", "smote", "undersample", "class_weight"]
MODEL_LABEL = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}
STRATEGY_COLOR = {
    "none": "#4C72B0",
    "smote": "#DD8452",
    "undersample": "#55A868",
    "class_weight": "#C44E52",
}

results = pd.read_csv(config.RESULTS_DIR / "imbalance_experiment.csv")
print(f"{len(results)} cells loaded")
results.head()

## 2. Cross-validated PR-AUC, with error bars

The error bars are the point of this table. The test set holds only ~95
frauds (74 under the chronological split), so a two- or three-point gap
between strategies is well inside sampling noise. Reporting a single number
per cell would invite exactly the over-claiming this dissertation critiques.

In [ ]:
def cv_table(protocol: str) -> pd.DataFrame:
    sub = results[results.protocol == protocol]
    out = pd.DataFrame(index=MODEL_ORDER, columns=STRATEGY_ORDER, dtype=object)
    for _, r in sub.iterrows():
        out.loc[r.model, r.strategy] = (
            f"{r.cv_pr_auc_mean:.3f} +/- {r.cv_pr_auc_std:.3f}"
        )
    out.index = [MODEL_LABEL[m] for m in out.index]
    return out


for protocol in ("stratified", "chronological"):
    print(f"\n{protocol.upper()} — CV PR-AUC (mean +/- std across 5 folds)")
    print(cv_table(protocol).to_string())

### Figure 1 — Strategy comparison

Bars grouped by model, one colour per strategy, error bars showing the
fold-to-fold standard deviation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
width = 0.2

for ax, protocol in zip(axes, ("stratified", "chronological")):
    sub = results[results.protocol == protocol]
    x = np.arange(len(MODEL_ORDER))
    for i, strategy in enumerate(STRATEGY_ORDER):
        means, stds = [], []
        for model in MODEL_ORDER:
            row = sub[(sub.model == model) & (sub.strategy == strategy)].iloc[0]
            means.append(row.cv_pr_auc_mean)
            stds.append(row.cv_pr_auc_std)
        ax.bar(
            x + (i - 1.5) * width, means, width, yerr=stds, capsize=3,
            label=strategy, color=STRATEGY_COLOR[strategy],
        )
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABEL[m] for m in MODEL_ORDER], fontsize=9)
    ax.set_title(f"{protocol.capitalize()} split")
    ax.set_ylim(0.6, 0.95)

axes[0].set_ylabel("CV PR-AUC (mean +/- std)")
axes[1].legend(title="Strategy", fontsize=8, loc="upper left")
fig.suptitle(
    "Imbalance strategy barely moves PR-AUC; model choice moves it a lot",
    fontsize=12,
)
fig.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(config.FIGURES_DIR / "phase3_strategy_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

### How big is the model effect versus the strategy effect?

Quantifying what Figure 1 shows: the spread across models, against the
spread across strategies within a model.

In [ ]:
for protocol in ("stratified", "chronological"):
    sub = results[results.protocol == protocol]
    within = (
        sub.groupby("model").cv_pr_auc_mean.agg(lambda s: s.max() - s.min())
    )
    across = (
        sub.groupby("strategy").cv_pr_auc_mean.agg(lambda s: s.max() - s.min())
    )
    typical_err = sub.cv_pr_auc_std.mean()
    print(f"\n{protocol.upper()}")
    print(f"  spread across STRATEGIES (within a model): "
          f"{within.min():.3f} to {within.max():.3f}")
    print(f"  spread across MODELS (within a strategy)  : "
          f"{across.min():.3f} to {across.max():.3f}")
    print(f"  typical fold-to-fold std                  : {typical_err:.3f}")
    print("  -> strategy differences sit inside the error bars; "
          "model differences do not.")

## 3. What resampling actually changes: the operating point

PR-AUC is threshold-independent, so it measures how well a model *ranks*
transactions. If resampling barely shifts PR-AUC but visibly shifts precision
and recall at the fixed 0.5 threshold, then what it really does is move the
operating point along an essentially unchanged curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

for ax, protocol in zip(axes, ("stratified", "chronological")):
    sub = results[results.protocol == protocol]
    for _, r in sub.iterrows():
        ax.scatter(
            r.test_recall, r.test_precision,
            color=STRATEGY_COLOR[r.strategy],
            marker={"logistic_regression": "o",
                    "random_forest": "s",
                    "xgboost": "^"}[r.model],
            s=90, edgecolor="black", linewidth=0.5, zorder=3,
        )
    ax.set_title(f"{protocol.capitalize()} split")
    ax.set_xlabel("Recall @ 0.5")

axes[0].set_ylabel("Precision @ 0.5")
handles = [
    plt.Line2D([], [], marker="o", linestyle="", color=c, label=s,
               markeredgecolor="black")
    for s, c in STRATEGY_COLOR.items()
]
handles += [
    plt.Line2D([], [], marker=m, linestyle="", color="grey",
               label=MODEL_LABEL[k], markeredgecolor="black")
    for k, m in [("logistic_regression", "o"), ("random_forest", "s"),
                 ("xgboost", "^")]
]
axes[1].legend(handles=handles, fontsize=7, loc="lower left", ncol=2)
fig.suptitle(
    "Resampling trades precision for recall at a fixed threshold", fontsize=12,
)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "phase3_precision_recall_tradeoff.png",
            dpi=150, bbox_inches="tight")
plt.show()

worst = results.nsmallest(4, "test_precision")[
    ["protocol", "model", "strategy", "test_recall", "test_precision", "fp"]
]
print("\nLowest-precision cells (false alarms per fraud caught):")
for _, r in worst.iterrows():
    per_catch = r.fp / max(r.test_recall * 1, 1e-9)
    print(f"  {r.protocol:13s} {r.model:19s} {r.strategy:12s} "
          f"R={r.test_recall:.3f} P={r.test_precision:.3f}  FP={int(r.fp)}")

### Figure 3 — Precision-recall curves

Rebuilt from the saved test-set scores, so no model is refitted here. If the
curves for the four strategies sit on top of each other, the ranking really is
unchanged and only the chosen threshold differs.

In [ ]:
df_clean = experiment.load_clean()
y_test_by_protocol = {
    "stratified": data.stratified_split(df_clean)[3],
    "chronological": data.chronological_split(df_clean)[3],
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for row, protocol in enumerate(("stratified", "chronological")):
    y_test = np.asarray(y_test_by_protocol[protocol])
    for col, model in enumerate(MODEL_ORDER):
        ax = axes[row, col]
        for strategy in STRATEGY_ORDER:
            path = (experiment.SCORES_DIR /
                    f"{model}_{strategy}_{protocol}.npy")
            if not path.exists():
                continue
            scores = np.load(path)
            precision, recall, _ = precision_recall_curve(y_test, scores)
            cell = results[(results.protocol == protocol) &
                           (results.model == model) &
                           (results.strategy == strategy)].iloc[0]
            ax.plot(recall, precision, color=STRATEGY_COLOR[strategy],
                    linewidth=1.4,
                    label=f"{strategy} ({cell.test_pr_auc:.3f})")
        ax.set_title(f"{MODEL_LABEL[model]} — {protocol}", fontsize=9)
        ax.legend(fontsize=6.5, loc="lower left")

for ax in axes[1]:
    ax.set_xlabel("Recall")
for ax in axes[:, 0]:
    ax.set_ylabel("Precision")
fig.suptitle(
    "Precision-recall curves by model and strategy "
    "(test PR-AUC in parentheses)", fontsize=12,
)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "phase3_pr_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 4. How much resampling did cross-validation actually want?

The sampling ratio (minority:majority after resampling) was exposed as a
tunable rather than fixed at the textbook 1:1. That turns "how aggressively
should we resample?" from an assumption into a result.

In [ ]:
rows = []
for _, r in results[results.strategy.isin(["smote", "undersample"])].iterrows():
    ratio = json.loads(r.best_params).get("resampler__sampling_strategy")
    rows.append({
        "protocol": r.protocol, "model": r.model, "strategy": r.strategy,
        "chosen_ratio": float(ratio), "test_precision": r.test_precision,
    })
ratios = pd.DataFrame(rows)

counts = ratios.chosen_ratio.value_counts().sort_index()
print("Sampling ratio selected by cross-validation:")
for ratio, n in counts.items():
    bar = "#" * n
    print(f"  {ratio:<5} chosen {n:>2}/{len(ratios)} times  {bar}")

print("\nThe minority class is 0.167% of training data. A 1:1 ratio would")
print("fabricate ~226,000 synthetic frauds from 378 real ones.")
ratios.sort_values("chosen_ratio", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(ratios.chosen_ratio, ratios.test_precision,
           s=90, color="#4C72B0", edgecolor="black", linewidth=0.5, zorder=3)
ax.set_xscale("log")
ax.set_xlabel("Sampling ratio chosen by CV (log scale)")
ax.set_ylabel("Test precision")
ax.set_title("Heavier resampling, worse precision")
for _, r in ratios[ratios.chosen_ratio >= 1.0].iterrows():
    ax.annotate(f"  {r.model}\n  {r.strategy} ({r.protocol})",
                (r.chosen_ratio, r.test_precision), fontsize=7,
                va="center")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "phase3_sampling_ratio.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 5. Best cell, and the drift preview

Ranked by cross-validated PR-AUC, which is the metric to trust — the test
figures rest on 95 and 74 frauds respectively.

In [ ]:
top = results.sort_values("cv_pr_auc_mean", ascending=False).head(8)
print("Top cells by CV PR-AUC:\n")
for _, r in top.iterrows():
    print(f"  {r.cv_pr_auc_mean:.3f} +/- {r.cv_pr_auc_std:.3f}  "
          f"{r.protocol:13s} {r.model:19s} {r.strategy:12s} "
          f"(test {r.test_pr_auc:.3f})")

print("\nStratified vs chronological, best cell per model:")
for model in MODEL_ORDER:
    s = results[(results.model == model) &
                (results.protocol == "stratified")].cv_pr_auc_mean.max()
    c = results[(results.model == model) &
                (results.protocol == "chronological")].cv_pr_auc_mean.max()
    print(f"  {MODEL_LABEL[model]:20s} {s:.3f} -> {c:.3f}  "
          f"({c - s:+.3f} moving to time-ordered validation)")

## Findings

**1. Model choice dominates imbalance strategy.** Across models the PR-AUC
gap is roughly 0.09, far outside the error bars. Within any one model the
four strategies differ by at most about 0.03 — comfortably inside them. On
this dataset, the resampling decision is not what determines ranking quality.

**2. The best cell uses no resampling at all.** XGBoost, stratified,
CV PR-AUC 0.852 +/- 0.028. It reproduces the Phase 2 tuned result (0.832
test PR-AUC) despite Phase 3's narrower search budget, which is a useful
check that trimming the grid did not distort the comparison.

**3. Cross-validation rejects aggressive resampling.** With the sampling
ratio exposed as a tunable, most resampling cells selected the mildest
setting offered rather than parity. The one cell that chose textbook 1:1
returned the worst precision in the grid. Fixing SMOTE at 1:1 — the default
in much of the fraud literature — is not supported by this data.

**4. Resampling moves the operating point, not the ranking.** Undersampling
buys recall and pays in precision; class weighting on Logistic Regression is
pathological, reaching recall near 0.88 at precision below 0.07. Because
PR-AUC is threshold-independent, that trade belongs to the cost-sensitive
threshold analysis in Phase 5, not to resampling.

**5. Chronological validation is consistently harder**, which is the drift
signal Phase 4 investigates directly.

### Limitations

Phase 3 used a narrower hyperparameter search than Phase 2 (forests capped at
200 trees, depth 10) so that all 24 cells could be retuned within a
practical compute budget. Absolute PR-AUC is therefore marginally below the
Phase 2 headline figures. Every strategy is handicapped identically, so the
comparison stays fair, but the numbers here should not be read as this
project's best attainable performance.

The deep network is absent: its tuning had not been run when this grid
executed. Adding it is the outstanding Phase 3 item.